# 03 — Feature Engineering

**Goals**
- Compute moisture trend (Δ over last 3 hourly readings)
- Extract hour-of-day (0–23)
- Build a single feature table
- Drop rows that lack a valid trend (first 3 hours of each cycle)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

manifest = pd.read_csv(PROCESSED_DIR / 'cycle_manifest.csv', parse_dates=['start_time', 'end_time'])
valid_ids = manifest[manifest['valid']]['cycle_id'].tolist()
print('Valid cycles:', valid_ids)


In [ ]:
# Load valid cycles and engineer features
feature_frames = []
for f in sorted(RAW_DIR.glob('*.csv')):
    df = pd.read_csv(f, parse_dates=['timestamp'])
    cid = df['cycle_id'].iloc[0]
    if cid not in valid_ids:
        continue
    df = df.sort_values('timestamp').reset_index(drop=True)

    # Moisture trend = current moisture − moisture 3 hours earlier
    df['moisture_trend'] = df['soil_moisture_pct'] - df['soil_moisture_pct'].shift(3)

    # Hour of day
    df['hour_of_day'] = df['timestamp'].dt.hour

    cols = ['timestamp', 'cycle_id', 'condition',
            'temperature_C', 'humidity_pct', 'soil_moisture_pct',
            'moisture_trend', 'light_lux', 'hour_of_day']
    feature_frames.append(df[cols])

features = pd.concat(feature_frames, ignore_index=True)
print(f'Total rows before dropping NaN trends: {len(features)}')
features.head(8)


In [ ]:
# Drop the first 3 rows of every cycle (no valid 3-hour trend yet)
features_clean = features.dropna(subset=['moisture_trend']).reset_index(drop=True)
print(f'Rows after drop: {len(features_clean)}')
print('\nMissing-value check:')
print(features_clean.isnull().sum())


In [ ]:
# Trend should be mostly negative (soil is drying)
print('Moisture trend stats:')
display(features_clean['moisture_trend'].describe())
print('\nBy condition:')
display(features_clean.groupby('condition')['moisture_trend'].describe())


In [ ]:
# Save intermediate feature table
features_clean.to_csv(PROCESSED_DIR / 'features.csv', index=False)
print('Saved → data/processed/features.csv')
